# Preparación de datos — Fase 3 de CRISP-DM

Proyecto Integrador · Minería de Datos · UNMSM FISI

Ejecuta la matriz de tratamiento presentada en el 1° avance y produce la tabla
maestra que consumen los seis módulos de modelamiento. Cubre los puntos 14 y 15
de la propuesta.

**Entrada:** `data/raw/*.csv` · **Salida:** `data/processed/master_pedidos.csv`

La lógica vive en `src/data/build_master.py`; este notebook la ejecuta, documenta
y valida.

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "02_preparation" else Path.cwd()
sys.path.insert(0, str(ROOT))
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

from src.data.build_master import (build_master, load_raw, dedup_geolocation,
                                   dedup_reviews, quality_report)
print("ROOT:", ROOT.resolve())

ROOT: C:\Users\Tutur\Desktop\CursoMineria\Olist-ecommerce-analytics


## 1. Carga de las fuentes

In [2]:
dfs = load_raw(ROOT / "data" / "raw")
pd.DataFrame({"tabla": list(dfs), "filas": [len(d) for d in dfs.values()],
              "columnas": [d.shape[1] for d in dfs.values()]})

,tabla,filas,columnas
0,customers,99441,5
1,geolocation,1000163,5
2,orders,99441,8
3,order_items,112650,7
4,payments,103886,5
5,reviews,99224,7
6,products,32951,9
7,sellers,3095,4
8,category_tr,71,2


## 2. Valores faltantes

No todo nulo es un dato perdido: la mayoría son ausencias estructurales que
imputar sería inventar información.

In [3]:
nulos = (pd.concat({t: d.isna().mean() * 100 for t, d in dfs.items()})
           .rename("%_nulos").reset_index()
           .rename(columns={"level_0": "tabla", "level_1": "variable"}))
nulos = nulos[nulos["%_nulos"] > 0].sort_values("%_nulos", ascending=False)

TRATAMIENTO = {
    "review_comment_title":          "Descartar: 88% ausente",
    "review_comment_message":        "No imputar -> flag tiene_comentario; el 41% restante alimenta NLP",
    "order_delivered_customer_date": "No imputar: el pedido aún no se entregó",
    "order_delivered_carrier_date":  "No imputar: estructural",
    "order_approved_at":             "No imputar: pago no aprobado",
    "product_category_name":         "Imputar 'sin_categoria'",
    "product_weight_g":              "Mediana por categoría",
}
nulos["tratamiento"] = nulos.variable.map(TRATAMIENTO).fillna("Mediana por categoría")
nulos.reset_index(drop=True)

,tabla,variable,%_nulos,tratamiento
0,reviews,review_comment_title,88.34,Descartar: 88% ausente
1,reviews,review_comment_message,58.70,No imputar -> flag tiene_comentario; el 41% re...
2,orders,order_delivered_customer_date,2.98,No imputar: el pedido aún no se entregó
3,products,product_category_name,1.85,Imputar 'sin_categoria'
4,products,product_name_lenght,1.85,Mediana por categoría
5,products,product_description_lenght,1.85,Mediana por categoría
6,products,product_photos_qty,1.85,Mediana por categoría
7,orders,order_delivered_carrier_date,1.79,No imputar: estructural
8,orders,order_approved_at,0.16,No imputar: pago no aprobado
9,products,product_weight_g,0.01,Mediana por categoría


Comprobación de que los nulos de entrega son estructurales: todo pedido sin
fecha debe tener un estado distinto de `delivered`, salvo los 8 casos
inconsistentes conocidos.

In [4]:
o = dfs["orders"]
pd.crosstab(o.order_status, o.order_delivered_customer_date.isna(),
            margins=True, colnames=["sin fecha de entrega"])

sin fecha de entrega,False,True,All
order_status,,,
approved,0,2,2
canceled,6,619,625
created,0,5,5
delivered,96470,8,96478
invoiced,0,314,314
processing,0,301,301
shipped,0,1107,1107
unavailable,0,609,609
All,96476,2965,99441


## 3. Duplicados

| Tabla | Duplicados | Tratamiento |
|---|---|---|
| `geolocation` | 261,831 filas exactas | Centroide por prefijo postal |
| `reviews` | 814 `review_id`; 547 pedidos con >1 | Conservar la más reciente |
| `orders`, `customers`, `products` | 0 | Integridad correcta |

In [5]:
antes_geo, antes_rev = len(dfs["geolocation"]), len(dfs["reviews"])
geo = dedup_geolocation(dfs["geolocation"])
rev = dedup_reviews(dfs["reviews"])

pd.DataFrame([
    {"tabla": "geolocation", "antes": antes_geo, "después": len(geo), "eliminados": antes_geo - len(geo)},
    {"tabla": "reviews",     "antes": antes_rev, "después": len(rev), "eliminados": antes_rev - len(rev)},
])

,tabla,antes,después,eliminados
0,geolocation,1000163,19015,981148
1,reviews,99224,98673,551


## 4. Integración y construcción de variables

Las definiciones son idénticas a las del notebook de EDA, para que las cifras del
1° avance sigan siendo reproducibles.

In [6]:
df = build_master(dfs)
print(f"{len(df):,} filas × {df.shape[1]} columnas")
print(f"Sin multiplicación de filas: {len(df) == len(dfs['orders'])}")
df.head(3)

99,441 filas × 51 columnas
Sin multiplicación de filas: True


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,n_items,n_productos,n_sellers,n_categorias,price_total,freight_total,categoria_principal,seller_state,peso_total_g,payment_value,n_cuotas,payment_type,review_score,review_comment_message,dias_entrega,dias_estimados,dias_retraso,dias_aprobacion,entrega_tardia,mismo_estado,ratio_flete,ticket,tiene_comentario,review_negativa,fecha,mes,anio,dia_semana,es_fin_semana,hora,flag_sin_items,flag_sin_pago,flag_pago_cero,flag_entregado_sin_fecha,flag_cancelado_con_entrega,flag_entrega_antes_transportista,flag_transportista_antes_aprobacion,flag_pago_descuadrado,registro_valido
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.00,1.00,1.00,1.00,29.99,8.72,housewares,SP,500.00,38.71,1.00,voucher,4.00,"Não testei o produto ainda, mas ele veio corre...",8.44,15.54,-7.11,0.01,0.00,1,0.29,38.71,1,0.00,2017-10-02,2017-10,2017,0,0,10,0,0,0,0,0,0,0,0,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.00,1.00,1.00,1.00,118.70,22.76,perfumery,SP,400.00,141.46,1.00,boleto,4.00,Muito bom o produto.,13.78,19.14,-5.36,1.28,0.00,0,0.19,141.46,1,0.00,2018-07-24,2018-07,2018,1,0,20,0,0,0,0,0,0,0,0,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.00,1.00,1.00,1.00,159.90,19.22,auto,SP,420.00,179.12,3.00,credit_card,5.00,NaN,9.39,26.64,-17.25,0.01,0.00,0,0.12,179.12,0,0.00,2018-08-08,2018-08,2018,2,0,8,0,0,0,0,0,0,0,0,1


## 5. Inconsistencias

No se borra nada en silencio: cada inconsistencia se marca con una bandera y cada
módulo decide qué excluir.

In [7]:
flags = [c for c in df.columns if c.startswith("flag_")]
resumen = pd.DataFrame({"bandera": flags, "registros": [int(df[c].sum()) for c in flags]})
resumen["%"] = (resumen.registros / len(df) * 100).round(3)

CRITERIO = {
    "flag_sin_items":                      "Excluir del análisis de ventas",
    "flag_sin_pago":                       "Excluir",
    "flag_pago_cero":                      "Excluir (9 filas de pago -> 3 pedidos con total 0)",
    "flag_entregado_sin_fecha":            "Excluir del cálculo de días de entrega",
    "flag_cancelado_con_entrega":          "Reclasificar",
    "flag_entrega_antes_transportista":    "Error de registro: excluir",
    "flag_transportista_antes_aprobacion": "Conservar: aprobación tardía del pago",
    "flag_pago_descuadrado":               "Conservar: voucher y redondeo lo explican",
}
resumen["criterio"] = resumen.bandera.map(CRITERIO)
resumen

,bandera,registros,%,criterio
0,flag_sin_items,775,0.78,Excluir del análisis de ventas
1,flag_sin_pago,1,0.00,Excluir
2,flag_pago_cero,3,0.00,Excluir (9 filas de pago -> 3 pedidos con tota...
3,flag_entregado_sin_fecha,8,0.01,Excluir del cálculo de días de entrega
4,flag_cancelado_con_entrega,6,0.01,Reclasificar
5,flag_entrega_antes_transportista,23,0.02,Error de registro: excluir
6,flag_transportista_antes_aprobacion,1359,1.37,Conservar: aprobación tardía del pago
7,flag_pago_descuadrado,312,0.31,Conservar: voucher y redondeo lo explican


In [8]:
print(f"Registros válidos: {df.registro_valido.sum():,} de {len(df):,} "
      f"({df.registro_valido.mean() * 100:.2f}%)")

Registros válidos: 98,629 de 99,441 (99.18%)


## 6. Outliers — IQR y Z-Score

No se eliminan. Un 13.8% de outliers en peso no son errores: son productos
legítimamente pesados. Para árboles se conservan; para modelos lineales y de
distancia se winsoriza al p99 o se aplica log.

In [9]:
def outliers_iqr(s):
    s = s.dropna()
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()), len(s)

def outliers_z(s, umbral=3):
    s = s.dropna()
    if s.std(ddof=0) == 0:
        return 0, len(s)
    return int((((s - s.mean()) / s.std(ddof=0)).abs() > umbral).sum()), len(s)

VARS = ["price_total", "freight_total", "payment_value", "peso_total_g", "dias_entrega"]
filas = []
for v in VARS:
    n_iqr, total = outliers_iqr(df[v])
    n_z, _ = outliers_z(df[v])
    filas.append({"variable": v, "IQR": n_iqr, "%_IQR": round(n_iqr / total * 100, 2),
                  "Z>3": n_z, "%_Z": round(n_z / total * 100, 2)})
pd.DataFrame(filas)

,variable,IQR,%_IQR,Z>3,%_Z
0,price_total,7913,8.02,1709,1.73
1,freight_total,9964,10.10,1570,1.59
2,payment_value,7866,7.91,1724,1.73
3,peso_total_g,13804,13.99,1962,1.99
4,dias_entrega,4899,5.08,1571,1.63


IQR marca más que Z-Score porque estas distribuciones son asimétricas, no
normales. IQR es el criterio válido; Z-Score se reporta como contraste.

In [10]:
def winsorizar(s, inf=0.01, sup=0.99):
    return s.clip(s.quantile(inf), s.quantile(sup))

pd.DataFrame({
    "original": df.freight_total.describe(),
    "winsorizado_p1_p99": winsorizar(df.freight_total).describe(),
    "log1p": np.log1p(df.freight_total).describe(),
}).round(2)

,original,winsorizado_p1_p99,log1p
count,"98,666.00","98,666.00","98,666.00"
mean,22.87,22.32,2.99
std,21.83,16.39,0.56
min,0.00,7.39,0.00
25%,13.85,13.85,2.70
50%,17.18,17.18,2.90
75%,24.06,24.06,3.22
max,"1,794.96",105.47,7.49


## 7. Desbalance de la variable objetivo

In [11]:
dist = df.review_score.value_counts(normalize=True).sort_index() * 100
neg = df.review_negativa.mean() * 100
print(dist.round(2).to_string())
print(f"\nNegativa (1-2*): {neg:.2f}%   Resto: {100 - neg:.2f}%   ratio {(100 - neg) / neg:.1f} : 1")

review_score
1.00   11.52
2.00    3.17
3.00    8.24
4.00   19.29
5.00   57.77

Negativa (1-2*): 14.69%   Resto: 85.31%   ratio 5.8 : 1


SMOTE se aplica solo al conjunto de entrenamiento y después de partir los datos.
Aplicarlo antes del split filtra registros sintéticos al test e infla las
métricas.

Con 14.69% de positivos accuracy no sirve: responder siempre "no es negativa"
acierta el 85%. Las métricas correctas son Recall, F1 y PR-AUC.

In [12]:
from sklearn.model_selection import train_test_split

base = df[df.review_score.notna() & df.dias_entrega.notna()]
X = base[["price_total", "freight_total", "n_items", "dias_entrega", "dias_retraso"]].fillna(0)
y = base.review_negativa.astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
print(f"ANTES   -> train {y_tr.mean() * 100:.2f}% positivos | test {y_te.mean() * 100:.2f}%")

try:
    from imblearn.over_sampling import SMOTE
    X_bal, y_bal = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
    print(f"DESPUÉS -> train {y_bal.mean() * 100:.2f}% ({len(X_bal):,} filas) | "
          f"test {y_te.mean() * 100:.2f}% sin tocar")
except ImportError:
    print("imbalanced-learn no instalado: pip install imbalanced-learn")

ANTES   -> train 12.81% positivos | test 12.81%


DESPUÉS -> train 50.00% (133,686 filas) | test 12.81% sin tocar


## 8. Codificación y escalamiento

Van dentro del pipeline de cada módulo, no en la tabla compartida: el `fit` debe
hacerse solo con datos de entrenamiento.

In [13]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder, StandardScaler

cats = df[["payment_type", "customer_state"]].fillna("desconocido")
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit_transform(cats[["payment_type"]])
print("One-Hot ->", ohe.shape, "| categorías:", cats.payment_type.nunique())
print("Label   ->", LabelEncoder().fit_transform(cats.customer_state)[:8],
      f"... ({cats.customer_state.nunique()} estados)")

num = df[["price_total", "freight_total", "dias_entrega"]].fillna(0)
print("\nStandardScaler -> media", np.round(StandardScaler().fit_transform(num).mean(0), 3))
mm = MinMaxScaler().fit_transform(num)
print("MinMaxScaler   -> rango", np.round(mm.min(0), 3), np.round(mm.max(0), 3))

One-Hot -> (99441, 6) | categorías: 6
Label   -> [25  4  8 19 25 17 22 18] ... (27 estados)

StandardScaler -> media [-0.  0. -0.]
MinMaxScaler   -> rango [0. 0. 0.] [1. 1. 1.]


## 9. Colinealidad y reducción de dimensionalidad

In [14]:
NUM = ["price_total", "freight_total", "ratio_flete", "n_items", "n_sellers",
       "peso_total_g", "n_cuotas", "dias_entrega", "dias_estimados", "dias_retraso", "ticket"]
corr = df[NUM].corr(method="spearman")

alta = (corr.where(~np.eye(len(corr), dtype=bool)).abs().stack()
            .loc[lambda s: s > 0.8].sort_values(ascending=False)
            .drop_duplicates().rename("|rho|").reset_index())
alta.columns = ["variable_a", "variable_b", "|rho|"]
print("Pares con |rho| > 0.8 (candidatos a eliminar):")
alta

Pares con |rho| > 0.8 (candidatos a eliminar):


,variable_a,variable_b,|rho|
0,price_total,ticket,0.99


In [15]:
from sklearn.decomposition import PCA

X_pca = StandardScaler().fit_transform(df[NUM].fillna(df[NUM].median()))
var = np.cumsum(PCA().fit(X_pca).explained_variance_ratio_)
n90 = int(np.argmax(var >= 0.90) + 1)
print(f"{n90} componentes explican el 90% de la varianza (de {len(NUM)} variables)")
pd.DataFrame({"componente": range(1, len(var) + 1), "varianza_acumulada": np.round(var, 3)}).head(8)

7 componentes explican el 90% de la varianza (de 11 variables)


,componente,varianza_acumulada
0,1,0.28
1,2,0.43
2,3,0.57
3,4,0.69
4,5,0.78
5,6,0.85
6,7,0.92
7,8,0.97


## 10. Guardar la tabla procesada

In [16]:
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

reporte = quality_report(df)
df.to_csv(PROCESSED / "master_pedidos.csv", index=False)
with (PROCESSED / "reporte_calidad.json").open("w", encoding="utf-8") as f:
    json.dump(reporte, f, ensure_ascii=False, indent=2)

print(f"Guardado: {PROCESSED / 'master_pedidos.csv'}")
print(f"{reporte['filas']:,} filas × {reporte['columnas']} columnas")
print(f"Periodo: {reporte['periodo'][0]} a {reporte['periodo'][1]}")

Guardado: C:\Users\Tutur\Desktop\CursoMineria\Olist-ecommerce-analytics\data\processed\master_pedidos.csv
99,441 filas × 51 columnas
Periodo: 2016-09-04 a 2018-10-17


## 11. Validación contra el 1° avance

Si la preparación cambiara los indicadores presentados, el código dejaría de
coincidir con lo expuesto.

In [17]:
h = json.load(open(ROOT / "docs" / "hallazgos_eda.json", encoding="utf-8"))
d = df[df.order_status == "delivered"].dropna(subset=["dias_entrega", "review_score"])

checks = [
    ("Reseñas negativas %",  round(df.review_negativa.mean() * 100, 2), h["review_negativa_%(<=2)"]),
    ("% entregas tardías",   round(d.entrega_tardia.mean() * 100, 2), h["%_entregas_tardias"]),
    ("MAE estimación Olist", round((d.dias_estimados - d.dias_entrega).abs().mean(), 2), h["MAE_estimacion_olist_dias"]),
    ("Días entrega mediana", round(d.dias_entrega.median(), 2), h["dias_entrega_mediana"]),
    ("% negativas a tiempo", round(d[d.entrega_tardia == 0].review_negativa.mean() * 100, 2), h["%_negativas_a_tiempo"]),
    ("% negativas tardías",  round(d[d.entrega_tardia == 1].review_negativa.mean() * 100, 2), h["%_negativas_tardias"]),
    ("Ticket promedio BRL",  round(df.loc[df.order_status != "canceled", "payment_value"].mean(), 2), h["ticket_promedio_BRL"]),
]
val = pd.DataFrame(checks, columns=["indicador", "tabla_maestra", "EDA_1er_avance"])
val["coincide"] = (val.tabla_maestra - val.EDA_1er_avance).abs() < 0.02
print("TODAS COINCIDEN" if val.coincide.all() else "REVISAR: hay diferencias")
val

TODAS COINCIDEN


,indicador,tabla_maestra,EDA_1er_avance,coincide
0,Reseñas negativas %,14.69,14.69,True
1,% entregas tardías,7.99,7.99,True
2,MAE estimación Olist,12.73,12.73,True
3,Días entrega mediana,10.21,10.21,True
4,% negativas a tiempo,9.22,9.22,True
5,% negativas tardías,54.07,54.07,True
6,Ticket promedio BRL,160.56,160.56,True


## Conclusiones

1. 99,441 pedidos integrados desde 9 tablas, sin multiplicación de filas.
2. 99.18% de registros válidos; las inconsistencias se marcan, no se borran.
3. Los nulos de entrega quedan demostrados como estructurales.
4. Los outliers se conservan: son negocio real, no errores.
5. Las 7 cifras del 1° avance se reproducen exactamente.

### Limitaciones

- El dataset carece de navegación, marketing, atención y devoluciones (4 de las 11
  fuentes de la propuesta). Churn se sustituye por reseña negativa.
- La recompra es de 3.06%, lo que limita RFM y cualquier modelo de abandono.
- Codificación, escalamiento y balanceo no se aplican aquí: van en el pipeline de
  cada módulo, ajustados solo con datos de entrenamiento.

### Punto de partida de cada módulo

```python
df = pd.read_csv("data/processed/master_pedidos.csv", parse_dates=["order_purchase_timestamp"])
df = df[df.registro_valido == 1]
```